# 03_03_Building `Fact_Punctuality`

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PATH_INTERMEDIATE = Path("../data/intermediate/")
PATH_PROCESSED = Path("../data/processed/")

# Table of Contents
- [3. FACT TABLE PUNCTUALITY](#3-fact-table-punctuality)

- [3.1. Adding `SK_Train_Service` and Optimising Memory Usage](#31-adding-sk_train_service-and-optimising-memory-usage)

- [3.2. Calculating New Delay Columns](#32-calculating-new-delay-columns)
    - [3.2.1. Converting `string` Time Columns to `timedelta64`](#321-converting-string-time-columns-to-timedelta64)
    - [3.2.2. Building Intermediate `timedelta64` Columns](#322-building-intermediate-timedelta64-columns)
    - [3.2.3. Computing New Delay Columns](#323-computing-new-delay-columns)

- [3.3. Detecting Date and Time Anomalies](#33-detecting-date-and-time-anomalies)
    - [3.3.1. Detecting Rows with Impossible Delay Calculations](#331-detecting-rows-with-impossible-delay-calculations)
    - [3.3.2. Detecting Inconsistent Rows](#332-detecting-inconsistent-rows)
        - [3.3.2.1. Detecting Unrealistic Time Periods](#3321-detecting-unrealistic-time-periods)
        - [3.3.2.2. Detecting Planned Date Inconsistencies](#3322-detecting-planned-date-inconsistencies)
    - [3.3.3. Checking for Anomalies in Years](#333-checking-for-anomalies-in-years)
    - [3.3.4. Dropping Intermediate `timedelta64` Columns](#334-dropping-intermediate-timedelta64-columns)

- [3.4. Enriching `punctuality` with `passengers_avg_sncb_2024_cleaned`](#34-enriching-punctuality-with-passengers_avg_sncb_2024_cleaned)
    - [3.4.1. Preparing `passengers` DataFrame](#341-preparing-passengers-dataframe)
    - [3.4.2. Merging `punctuality` and `passengers`](#342-merging-punctuality-and-passengers)
    - [3.4.3. Adding `Has_Passenger_Count` Flag Column and Dropping Columns after Merge](#343-adding-has_passenger_count-flag-column-and-dropping-columns-after-merge)

- [3.5. Adding `SK_Station`](#35-adding-sk_station)

- [3.6. Adding Flag Columns and `Weigthed_Delay`, and Preparing the Load to SQL Server](#36-adding-flag-columns-and-weigthed_delay-and-preparing-the-load-to-sql-server)
    - [3.6.1. Renaming and Converting Delay Columns to `Int64`](#361-renaming-and-converting-delay-columns-to-int64)
    - [3.6.2. Adding Flag Delay Columns](#362-adding-delay-flag-columns)
    - [3.6.3. Calculating `Weighted_Delay_Arr` and `Weighted_Delay_Dep`](#363-calculating-weighted_delay_arr-and-weighted_delay_dep)
    - [3.6.4. Converting `DATDEP` and Time Columns to `Int64`](#364-converting-datdep-and-time-columns-to-int64)

- [3.7. Refining `Fact_Punctuality`](#37-refining-fact_punctuality)

- [3.8. Exporting to Gold Layer](#38-exporting-to-gold-layer)


# 3. FACT TABLE PUNCTUALITY

**DESCRIPTION**

`Fact_Punctuality` is the **fact table** of the **star schema data warehouse** of this project. It is built from the Infrabel Open Data punctuality raw datasets, enriched by the SNCB 2024 average passenger count dataset.  

It spans the entire **2024**-**2025** time period.  

Its grain is an **individual train passage** at a station, identified by the combination of the train ID, the station, the service date, and the specific time of passage captured down to the second (see **NOTE: Definition of train passage** below).  

It has **45,542,079 entries**.  

`Fact_Punctuality` stores both original and recalculated arrival and departure **delay columns**. To support the project's two analytical hypotheses (see **README**), it includes **new intermediate delay measures** expressed in **passenger-seconds**, as well as **new binary indicators**. These binary columns flag delayed trains against two distinct thresholds: Infrabel's original six-minute rule and a stricter alternative five-minute threshold used to test the sensitivity of the punctuality analysis to the choice of delay definition.  

From `Fact_Punctuality`, **two VIEWS** will be created in SQL Server: `vw_arr_delay` and `vw_dep_delay`. Each view will focus on either arrival or departure date, time and delay measures. For this analysis, only `vw_arr_delay` will be loaded into Power BI (see **MODELLING DECISION 2** below).  

Once it is loaded into SQL Server, `Fact_Punctuality` will be linked to `Dim_Station`, `Dim_Train_Service`, `Dim_Date`, and `Dim_Time`.  

<br>

**NOTE: Definition of train passage**  
A train passage is both an arrival and a departure of a train on a station. Both arrival and departure times are usually recorded for each row, **except in two cases**:  
1) A train **starting its route** at its origin terminus has departure date and time but no arrival date and time.  

2) A train **ending its route** at its destination terminus has arrival date and time but no departure date and time.   

For this structural reason, departure and arrival date and time columns have **missing values**.  

<br>

**MODELLING DECISION: Two SQL views**

Train delay has **two possible definitions**: 
1) The difference between planned and real dates and times when the train **arrives at a station**.
2) The difference between planned and real dates and times when the train **departs from a station**.  

`Fact_Punctuality` contains the required data for both definitions, but for any given analysis, it can only be based on either one without adding ambiguity about the meaning of delay, and overcomplicating the links between the fact table and the time dimension.  

* From `Fact_Punctuality`, two views will be created in SQL Server, one for departure delays - `vw_dep_delay` - and the other for arrival delays - `vw_arr_delay`.  

* In Power BI, only `vw_arr_delay` will be loaded for data analysing and for testing the project analytical hypotheses, as established in the **README Overview** (*A train is considered late if it ***arrives*** more than 5 minutes after its scheduled arrival time*).


**ATTRIBUTES** 

`Fact_Punctuality` has **26** attributes.  

* Three **foreign keys** to dimensions:  
     - `Date_Service` (Int64): Train service date. Foreign key to `Dim_Date`.  
     - `SK_Station` (int64): Foreign key to `Dim_Station` with no missing values.  
     - `SK_Train_Service` (Int64): Foreign key to `Dim_Train_Service`.   

* The **train ID**: `Train_No` (Int64).  

* Four columns containing planned and real arrival and departure time at station, also **foreign keys** to `Dim_Time`: `Planned_Time_Arr`, `Real_Time_Arr`, `Planned_Time_Dep`, and `Real_Time_Dep` (Int64).  

* Six **delay columns**:  
    - Two new arrival and departure delay columns, calculated in seconds and built in this notebook: `Delay_Arr` and `Delay_Dep` (Int64).   
    - The original Infrabel delay columns, kept as references: `Delay_Arr_Infrabel` and `Delay_Dep_Infrabel` (Int64).  
    - `Weighted_Delay_Arr` and `Weighted_Delay_Dep` (Int64): Arrival and departure delays in seconds multiplied by the average passenger count (passenger-seconds). Required to calculate the final passenger-weighted delay metric of this analysis, whose calculation will be performed in Power BI (DAX) (see **README**).  

* Four **binary columns that flag trains** delayed according to either the original Infrabel delay metric or the stricter five-minute delay threshold: `Is_6Min_Late_Arr`, `Is_5Min_Late_Arr`, `Is_6Min_Late_Dep`, and `Is_5Min_Late_Dep` (Int64).  

* Two columns related to the **SNCB passenger count**: `Passenger_Count_Avg` (Int64), containing the average passenger count for a station, and `Has_Passenger_Count` (Int64), a binary column that flags rows whose `SK_Station` has a matching passenger count average in the SNCB source dataset.   

* Two railway line identifier columns - `Line_No_Arr` and `Line_No_Dep` (string). Although they are not used in the current analysis, they are included in the data warehouse for possible future use cases.  

* Four columns containing planned and real arrival or departure date at station. Used in combination with time columns to calculate `Delay_Arr` and `Delay_Dep` in this notebook. Kept as **timestamp columns**: `Planned_Date_Arr`, `Real_Date_Arr`, `Planned_Date_Dep`, and `Real_Date_Dep` (datetime64[us]).  

**NOTE**:   
* During the building of `Fact_Punctuality`, new derived `Delay_Arr` and `Delay_Dep` columns were recalculated for two reasons:
    - Logical inconsistencies revealed in the *02_04_profiling_and_cleaning_punctuality* notebook (Section 4.2.4.) for both Infrabel columns.
    - To ensure better control over the data.  
 
 <br>

**SOURCES** 

For `Fact_Punctuality`, the raw source datasets are:  

* The 24 monthly `punctuality_raw_MM_yyyy.csv` files from the Infrabel Open Data portal for 2024-2025.  

* `passengers_avg_sncb_2024_cleaned.parquet`, extracted from the SNCB PDF `voyageurs-montes-2024-def-1`.  

<br>

**BUILDING PROCESS** 

1) `punctuality_cleaned` dataset is loaded as `punctuality` DataFrame. Given that it contains more than 45 million entries and easily causes memory errors, the first building step is to add the surrogate key from `Dim_Train_Service` to `punctuality`, and then drop the four `string` columns from which this dimension was built (**Section 3.1.**).

2) New arrival and departure delay columns are calculated from the original Infrabel date and time columns. In this calculation, date and time columns are combined into intermediate `timedelta64` columns to correctly handle **trains crossing midnight** before the final delay computation (**Section 3.2.**).  

3) A series of logical rules are applied to `punctuality` as a **pipeline safeguard**. Any row violating these rules blocks the pipeline. **91 inconsistent rows** whose planned dates were incorrectly encoded are identified and excluded from the analysis.  (**Section 3.3.**).  

4) To enable calculation of the future **passenger-weighted metric**, `punctuality` is enriched with the SNCB passenger count average dataset and `passenger_average` is added from `passengers` to `punctuality`. In addition, a binary column is created to flag rows whose stations have no match in `passengers` (**Section 3.4.**).  

5) `punctuality` and `Dim_Station` are merged to add `SK_Station` to the fact table (**Section 3.5.**).  

6) New binary columns are created to flag trains delayed on arrival or departure, according to two different thresholds. Then, `Weighted_Delay` is calculated by multiplying the delay columns by `passenger_average`. In addition, service date, delay, and time columns are converted to `Int64`, preparing the load to SQL Server (**Section 3.6.**).  

7) Columns are renamed and reorganised for better clarity. Finally, `punctuality` is exported as `Fact_Punctuality` to the `processed` directory. The result of the export is verified (**Sections 3.7. and 3.8.**).

<br>

In [3]:
punctuality = pd.read_parquet(PATH_INTERMEDIATE / "punctuality_cleaned.parquet")
punctuality.head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED
0,2024-01-01,1813,IC 02,SNCB/NMBS,929,50A,<NA>,13:09:42,<NA>,13:09:00,NaN,42.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,<NA>,NaT,2024-01-01,NaT,2024-01-01,0
1,2024-01-01,1813,IC 02,SNCB/NMBS,210,50A,13:22:17,13:25:20,13:22:00,13:25:00,17.0,20.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0
2,2024-01-01,1813,IC 02,SNCB/NMBS,931,50A,13:29:18,13:29:18,13:28:00,13:28:00,78.0,78.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0
3,2024-01-01,1813,IC 02,SNCB/NMBS,127,50A,13:31:59,13:31:59,13:31:00,13:31:00,59.0,59.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0
4,2024-01-01,1813,IC 02,SNCB/NMBS,797,50A,13:33:57,13:33:57,13:33:00,13:33:00,57.0,57.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0


In [4]:
punctuality.info()

<class 'pandas.DataFrame'>
RangeIndex: 45542084 entries, 0 to 45542083
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   DATDEP                 datetime64[us]
 1   TRAIN_NO               Int64         
 2   RELATION               string        
 3   TRAIN_SERV             string        
 4   PTCAR_NO               Int64         
 5   LINE_NO_DEP            string        
 6   REAL_TIME_ARR          string        
 7   REAL_TIME_DEP          string        
 8   PLANNED_TIME_ARR       string        
 9   PLANNED_TIME_DEP       string        
 10  DELAY_ARR              float64       
 11  DELAY_DEP              float64       
 12  RELATION_DIRECTION     string        
 13  PTCAR_LG_NM_NL         string        
 14  LINE_NO_ARR            string        
 15  PLANNED_DATE_ARR       datetime64[us]
 16  PLANNED_DATE_DEP       datetime64[us]
 17  REAL_DATE_ARR          datetime64[us]
 18  REAL_DATE_DEP          datetime

## 3.1. Adding `SK_Train_Service` and Optimising Memory Usage

The `punctuality` DataFrame contains 45 million rows and has a memory usage of **10.8 GB**. This easily causes **memory errors** with a standard 16 GB RAM machine. For this reason, the first section of this notebook drops four `string` columns and reduces the `punctuality` memory usage. 

* `Dim_Train_Service` is merged into `punctuality` to add `SK_Train_Service`.

* Before the merge, the irrelevant `Dim_Train_Service` columns are dropped: `Relation_Category`, `Is_Local`, and `Is_National`. 

* The number of `punctuality` rows is checked before and after the merge.  

* Once `SK_Train_Service` is merged to `punctuality`, the redundant columns created by the merge and the four `string` columns from which `Dim_Train_Service` was built are dropped. 

* The `punctuality` memory usage is now **7.8 GB**.

In [5]:
dim_train_service = pd.read_parquet(PATH_PROCESSED / "Dim_Train_Service.parquet")

In [6]:
dim_train_service = (
    dim_train_service.drop(columns=["Relation_Category", "Is_Local", "Is_National"])
)

In [7]:
punctuality_length = len(punctuality)

In [8]:
punctuality = punctuality.merge(
    dim_train_service,
    left_on=["TRAIN_SERV", "RELATION", "RELATION_DIRECTION", "DIRECTION_IS_INFERRED"],
    right_on=["Operator_Train_Service", "Relation", "Relation_Direction", "Direction_Is_Inferred"],
    how="left",
    indicator=True
)

In [9]:
assert len(punctuality) == punctuality_length

In [10]:
punctuality["_merge"].value_counts()

_merge
both          45542084
left_only            0
right_only           0
Name: count, dtype: int64

In [11]:
punctuality = (
    punctuality.drop(columns=["RELATION", "TRAIN_SERV", "RELATION_DIRECTION", "DIRECTION_IS_INFERRED",
                              "Operator_Train_Service", "Relation", "Relation_Direction", 
                              "Direction_Is_Inferred", "_merge"])
)

In [12]:
del dim_train_service

In [13]:
punctuality.memory_usage(deep=True).sum() / 1024**3

np.float64(7.813784509897232)

## 3.2. Calculating New Delay Columns

* As established in the *02_04_profiling_and_cleaning_punctuality* notebook (Section 4.2.4.), there are inconsistencies in the original Infrabel delay columns. Therefore, for **better control over the data**, the arrival and departure delay columns are **recalculated** in the section below, and temporarily named `New_Delay_Arrival` and `New_Delay_Departure`.    

* Their calculation is the following:
    - `New_Delay_Arrival` = (`PLANNED_DATE_ARR` + `PLANNED_TIME_ARR`) - (`REAL_DATE_ARR` + `REAL_TIME_ARR`)  
    - `New_Delay_Departure` = (`PLANNED_DATE_DEP` + `PLANNED_TIME_DEP`) - (`REAL_DATE_DEP` + `REAL_TIME_DEP`)  

In **Section 3.2.1.**:  
* Before the calculation, the time columns are converted from `string` to `timedelta64[s]`.  

* The number of missing values in arrival and departure columns is checked before and after the conversion to verify that `errors="coerce"` does not create new `NaT` values from incorrectly formated `string` values.  

In **Section 3.2.2.**:  
* Temporary `timedelta64` columns are built by combining date and time columns, according to the following rules:  
    - `Real_Arrival` = `REAL_DATE_ARR` + `REAL_TIME_ARR`  
    - `Real_Departure` = `REAL_DATE_DEP` + `REAL_TIME_DEP`  
    - `Planned_Arrival` = `PLANNED_DATE_ARR` + `PLANNED_TIME_ARR`  
    - `Planned_Departure` = `PLANNED_DATE_DEP` + `PLANNED_TIME_DEP`  

* This intermediate columns correctly **handle trains crossing midnight**, where the real time exceeds the planned time on a different calendar day.

In **Section 3.2.3.**:  
* `New_Delay_Arrival` and `New_Delay_Departure` are calculated.  

### 3.2.1. Converting `string` Time Columns to `timedelta64`

In [14]:
cols_to_cast = ["PLANNED_TIME_ARR", "REAL_TIME_ARR", "PLANNED_TIME_DEP", "REAL_TIME_DEP"]

In [15]:
punctuality[cols_to_cast].dtypes

PLANNED_TIME_ARR    string
REAL_TIME_ARR       string
PLANNED_TIME_DEP    string
REAL_TIME_DEP       string
dtype: object

In [16]:
nan_in_time_cols = punctuality[cols_to_cast].isnull().sum()
nan_in_time_cols

PLANNED_TIME_ARR    2254669
REAL_TIME_ARR       2254669
PLANNED_TIME_DEP    2251535
REAL_TIME_DEP       2251535
dtype: int64

**⚠️ Warning**: The cell below may take **3 to 4 minutes** to execute.

In [17]:
punctuality[cols_to_cast] = punctuality[cols_to_cast].apply(pd.to_timedelta, errors="coerce")

In [18]:
for col in cols_to_cast:
    punctuality[col] = punctuality[col].astype("timedelta64[s]")

In [19]:
nan_timedelta_cols = punctuality[cols_to_cast].isnull().sum()
nan_timedelta_cols

PLANNED_TIME_ARR    2254669
REAL_TIME_ARR       2254669
PLANNED_TIME_DEP    2251535
REAL_TIME_DEP       2251535
dtype: int64

In [20]:
nan_in_time_cols.equals(nan_timedelta_cols)

True

No new `NaT` values.  

In [21]:
punctuality.dtypes

DATDEP              datetime64[us]
TRAIN_NO                     Int64
PTCAR_NO                     Int64
LINE_NO_DEP                 string
REAL_TIME_ARR       timedelta64[s]
REAL_TIME_DEP       timedelta64[s]
PLANNED_TIME_ARR    timedelta64[s]
PLANNED_TIME_DEP    timedelta64[s]
DELAY_ARR                  float64
DELAY_DEP                  float64
PTCAR_LG_NM_NL              string
LINE_NO_ARR                 string
PLANNED_DATE_ARR    datetime64[us]
PLANNED_DATE_DEP    datetime64[us]
REAL_DATE_ARR       datetime64[us]
REAL_DATE_DEP       datetime64[us]
SK_Train_Service             Int64
dtype: object

### 3.2.2. Building Intermediate `timedelta64` Columns

In [22]:
punctuality["Real_Departure"] = punctuality["REAL_DATE_DEP"] + punctuality["REAL_TIME_DEP"]
punctuality["Real_Arrival"] = punctuality["REAL_DATE_ARR"] + punctuality["REAL_TIME_ARR"]

In [23]:
punctuality["Planned_Departure"] = punctuality["PLANNED_DATE_DEP"] + punctuality["PLANNED_TIME_DEP"]
punctuality["Planned_Arrival"] = punctuality["PLANNED_DATE_ARR"] + punctuality["PLANNED_TIME_ARR"]

### 3.2.3. Computing New Delay Columns

In [24]:
punctuality["New_Delay_Arrival"] = punctuality["Real_Arrival"] - punctuality["Planned_Arrival"]
punctuality["New_Delay_Departure"] = punctuality["Real_Departure"] - punctuality["Planned_Departure"]

In [25]:
punctuality["New_Delay_Arrival"] = punctuality["New_Delay_Arrival"].astype("timedelta64[s]")
punctuality["New_Delay_Departure"] = punctuality["New_Delay_Departure"].astype("timedelta64[s]")

In [26]:
punctuality[["New_Delay_Arrival", "New_Delay_Departure"]].head()

,New_Delay_Arrival,New_Delay_Departure
0,NaT,0 days 00:00:42
1,0 days 00:00:17,0 days 00:00:20
2,0 days 00:01:18,0 days 00:01:18
3,0 days 00:00:59,0 days 00:00:59
4,0 days 00:00:57,0 days 00:00:57


## 3.3. Detecting Date and Time Anomalies

In the section below, a series of logical rules is applied to the date and time columns of `punctuality`.  

These rules act as a **pipeline safeguard**:  
* For each rule, any row violating the rule is dropped and hence a new version of `punctuality` is created.  
* Then, the lengths of the earlier and newer `punctuality` versions are compared. If the lengths are not the same, the pipeline is blocked.    

An exception exists in Section 3.3.2.1., given that **78 illogical rows were already identified and dropped** during pipeline creation. Therefore, after this section, for each following rule, the pipeline safeguard uses the `punctuality` length measured after this drop as its reference, instead of the original one.  

### 3.3.1. Detecting Rows with Impossible Delay Calculations

* To calculate a train delay, at least one real time column - arrival or departure - and at least one corresponding planned time column are required: 
    - `PLANNED_TIME_ARR` if the real time column is `REAL_TIME_ARR`,
    - and `PLANNED_TIME_DEP` if the real time column is `REAL_TIME_DEP`.  

* In all other cases, calculating the train delay is impossible. More precisely, if a row has missing values in both columns of any of the following pairs, computing the difference between the corresponding planned and real columns becomes impossible:    
    1) `REAL_TIME_DEP` and `REAL_TIME_ARR`   
    2) `PLANNED_TIME_ARR` and `PLANNED_TIME_DEP`   
    3) `REAL_TIME_DEP` and `PLANNED_TIME_ARR`     
    4) `REAL_TIME_ARR` and `PLANNED_TIME_DEP`   

* As a result, rows matching any of these four cases are dropped from `punctuality`.  

* After verification, no row matches any of these cases.  

In [27]:
rows_to_keep = (
    (punctuality["REAL_TIME_ARR"].notnull() & punctuality["PLANNED_TIME_ARR"].notnull())
    | (punctuality["REAL_TIME_DEP"].notnull() & punctuality["PLANNED_TIME_DEP"].notnull())
)
punctuality = punctuality[rows_to_keep]

In [28]:
assert len(punctuality) == punctuality_length

### 3.3.2. Detecting Inconsistent Rows

The section below contains safeguards for three distinct cases:  

1) **Unrealistic time periods**: Given the size of the Belgian railway network, no train route can span more than a single day. Hence, `PLANNED_DATE_DEP` or `PLANNED_DATE_ARR` cannot exceed `REAL_DATE_DEP` or `REAL_DATE_ARR` by more than 24 hours. In addition, the service date (`DATDEP`) cannot exceed `PLANNED_DATE_DEP`, `PLANNED_DATE_ARR`, `REAL_DATE_DEP`, or `REAL_DATE_ARR` by more than a day (Section 3.3.2.1.).  
 
    > The threshold is set to **72,000 seconds (20 hours)**. A threshold analysis showed that the number of 
    > flagged rows stabilises at 91 between 20 and 12 hours, then rises again below 12 hours as exceptional 
    > delays on international services enter the selection. A 20-hour threshold cleanly separates date 
    > encoding errors from real delays, as a one-day offset cumulated with a delay of up to ~4 hours 
    > produces values between 20 and 24 hours. 

2) **Planned date inconsistencies**: A train's scheduled departure from a station must logically occur after its scheduled arrival. Consequently, `PLANNED_DATE_DEP` cannot exceed `PLANNED_DATE_ARR` (Section 3.3.2.2.).  

### 3.3.2.1. Detecting Unrealistic Time Periods

In [29]:
THRESHOLD_SECONDS = 72000

date_diff_plan_real_arr = (punctuality["Planned_Arrival"] - punctuality["Real_Arrival"]).dt.total_seconds()
date_diff_plan_real_dep = (punctuality["Planned_Departure"] - punctuality["Real_Departure"]).dt.total_seconds()


invalid_planned_dates = (
    (date_diff_plan_real_arr.notnull() & (date_diff_plan_real_arr.abs() >= THRESHOLD_SECONDS))
    | (date_diff_plan_real_dep.notnull() & (date_diff_plan_real_dep.abs() >= THRESHOLD_SECONDS))
)

invalid_planned_dates.sum()


np.int64(91)

In [30]:
datdep_diff_date_real_arr = (punctuality["DATDEP"] - punctuality["REAL_DATE_ARR"]).dt.days
datdep_diff_date_real_dep = (punctuality["DATDEP"] - punctuality["REAL_DATE_DEP"]).dt.days
datdep_diff_date_plan_arr = (punctuality["DATDEP"] - punctuality["PLANNED_DATE_ARR"]).dt.days
datdep_diff_date_plan_dep = (punctuality["DATDEP"] - punctuality["PLANNED_DATE_DEP"]).dt.days

invalid_datdep = (
    (datdep_diff_date_real_arr.notnull() & (datdep_diff_date_real_arr.abs() > 1))
    | (datdep_diff_date_plan_arr.notnull() & (datdep_diff_date_plan_arr.abs() > 1))
    | (datdep_diff_date_real_dep.notnull() & (datdep_diff_date_real_dep.abs() > 1))
    | (datdep_diff_date_plan_dep.notnull() & (datdep_diff_date_plan_dep.abs() > 1))
)

invalid_datdep.sum()

np.int64(5)

In [31]:
idx_planned_dates = set(punctuality[invalid_planned_dates].index)
idx_datdep = set(punctuality[invalid_datdep].index)

print(idx_datdep.issubset(idx_planned_dates))

True


The 5 rows with anomalies in `DATDEP` are included in the 91 rows with anomalies in planned date and time columns.

**⚠️ Warning**: The cell below may take **4 to 5 minutes** to execute.

In [32]:
punctuality = punctuality.loc[~invalid_planned_dates].reset_index(drop=True)

In [33]:
punctuality_new_length = len(punctuality)

### 3.3.2.2. Detecting Planned Date Inconsistencies

In [34]:
planned_date_anomalies = (punctuality["PLANNED_DATE_DEP"] < punctuality["PLANNED_DATE_ARR"])

planned_date_anomalies.sum()

np.int64(0)

In [35]:
punctuality = punctuality.loc[~planned_date_anomalies].reset_index(drop=True)

In [36]:
assert punctuality_new_length == len(punctuality)

### 3.3.3. Checking for Anomalies in Years

* All the dates must be between 2024 and 2025.  

* However, **342** out of 45 million rows are not.  

* Given that the source datasets were ingested from the Infrabel Open Data portal using their month and year as reference, it is very unlikely that these datasets contain entries out of the targeted time period. The most likely explanation is that they are trains **crossing midnight at the end of the year**: either starting their routes at the end of 31 December 2023 and ending them in 2024, or starting them at the end of 31 December 2025 and ending them in 2026. In these two cases, `2023` and `2026` should be existing values in `PLANNED_DATE_ARR`, `REAL_DATE_ARR`, `PLANNED_DATE_DEP`, and `REAL_DATE_DEP`, even though their train service dates (`DATDEP`) are between 2024 and 2025.  

* Extending the tested years to 2023 and 2026 effectively includes 100% of the dataset. The 342 anomalies disappear.  

* If one of these 342 anomalies is neither on 31 December 2023 nor 1 January 2026, the SQL `Dim_Date` will easily detect it.  

In [37]:
years = [2024, 2025]

invalid_years = (~punctuality["DATDEP"].dropna().dt.year.isin(years) 
                 | ~punctuality["REAL_DATE_ARR"].dropna().dt.year.isin(years) 
                 | ~punctuality["REAL_DATE_DEP"].dropna().dt.year.isin(years) 
                 | ~punctuality["PLANNED_DATE_ARR"].dropna().dt.year.isin(years) 
                 | ~punctuality["PLANNED_DATE_DEP"].dropna().dt.year.isin(years) 
                 )

invalid_years.sum()


np.int64(342)

In [38]:
years_extended = [2023, 2024, 2025, 2026]

invalid_years_extended = (~punctuality["DATDEP"].dropna().dt.year.isin(years_extended) 
                 | ~punctuality["REAL_DATE_ARR"].dropna().dt.year.isin(years_extended) 
                 | ~punctuality["REAL_DATE_DEP"].dropna().dt.year.isin(years_extended) 
                 | ~punctuality["PLANNED_DATE_ARR"].dropna().dt.year.isin(years_extended) 
                 | ~punctuality["PLANNED_DATE_DEP"].dropna().dt.year.isin(years_extended) 
                 )

invalid_years_extended.sum()


np.int64(0)

### 3.3.4. Dropping Intermediate `timedelta64` Columns

Given that the new delay columns are calculated and anomalies in date and time columns are detected and handled, the intermediate `Real_Departure`, `Real_Arrival`, `Planned_Departure`, and `Planned_Arrival` columns are now redundant and are therefore dropped.

In [39]:
punctuality = (
    punctuality.drop(columns=["Real_Departure", "Real_Arrival", "Planned_Departure", "Planned_Arrival"])
)

## 3.4. Enriching `punctuality` with `passengers_avg_sncb_2024_cleaned`

* To enable calculation of the future passenger-weighted metric, `punctuality` is enriched with the SNCB 2024 passenger count average, loaded in the `passengers` DataFrame. 

* `passengers` is unpivoted on `ptcarid` using `.melt()`: the three passenger count columns - `Weekdays`, `Saturdays`, and `Sundays` - are consolidated into a single `passenger_avg` column, and a `Day_Type` column is added to identify the day type each count refers to.

* A `Day_Type` column is added to `punctuality`, calculated from `DATDEP` and mapped to the following values: `Weekdays`, `Saturdays`, and `Sundays`.

* `punctuality` and `passengers` are merged on `PTCAR_NO` and `Day_Type` with `ptcarid` and `Day_Type`.

* Once the result of the merge is verified, the redundant columns from the merge are dropped.

* The resulting DataFrame is now `punctuality_passengers`.  

### 3.4.1. Preparing `passengers` DataFrame

In [40]:
passengers = pd.read_parquet(PATH_INTERMEDIATE / "passengers_avg_sncb_2024_cleaned.parquet")
passengers.head()

,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday,ptcarid,resolution_method,original_name
0,AALST,5883,2220,1872,6,direct_match,NaN
1,AALST-KERREBROEK,15,<NA>,<NA>,104,direct_match,NaN
2,AALTER,2348,622,620,8,direct_match,NaN
3,AARSCHOT,5804,2243,1758,9,direct_match,NaN
4,AARSELE,26,<NA>,<NA>,10,direct_match,NaN


In [41]:
passengers = passengers.rename(columns={
    "passenger_avg_weekdays" : "Weekdays",
    "passenger_avg_saturday" : "Saturdays",
    "passenger_avg_sunday" : "Sundays"
})

In [42]:
passengers_unpivot = passengers.melt(
    id_vars="ptcarid",
    value_vars=["Weekdays", "Saturdays", "Sundays"],
    var_name="Day_Type",
    value_name="passenger_avg"
)

### 3.4.2. Merging `punctuality` and `passengers`

* `passengers` is merged into `punctuality`, adding the `passenger_avg` column to this DataFrame.

* The result of the merge is named `punctuality_passengers`. 

* The redundant DataFrames are deleted to free up memory. 

In [43]:
day_type_map = {0: "Weekdays", 1: "Weekdays", 2: "Weekdays",
                3: "Weekdays", 4: "Weekdays", 5: "Saturdays", 6: "Sundays"}

punctuality["Day_Type"] = punctuality["DATDEP"].dt.day_of_week.map(day_type_map)

In [44]:
punctuality_passengers = punctuality.merge(
    passengers_unpivot,
    left_on=["PTCAR_NO", "Day_Type"],
    right_on=["ptcarid", "Day_Type"],
    how="left",
    indicator=True
)

In [45]:
assert len(punctuality_passengers) == punctuality_new_length

In [46]:
punctuality_passengers["_merge"].value_counts()

_merge
both          42610066
left_only      2931927
right_only           0
Name: count, dtype: int64

In [47]:
del punctuality
del passengers
del passengers_unpivot

### 3.4.3. Adding `Has_Passenger_Count` Flag Column and Dropping Columns after Merge

`Has_Passenger_Count` is created.  
It is a binary column that flags rows whose station has an average passenger count in the SNCB 2024 source dataset.  

In [48]:
punctuality_passengers.columns

Index(['DATDEP', 'TRAIN_NO', 'PTCAR_NO', 'LINE_NO_DEP', 'REAL_TIME_ARR',
       'REAL_TIME_DEP', 'PLANNED_TIME_ARR', 'PLANNED_TIME_DEP', 'DELAY_ARR',
       'DELAY_DEP', 'PTCAR_LG_NM_NL', 'LINE_NO_ARR', 'PLANNED_DATE_ARR',
       'PLANNED_DATE_DEP', 'REAL_DATE_ARR', 'REAL_DATE_DEP',
       'SK_Train_Service', 'New_Delay_Arrival', 'New_Delay_Departure',
       'Day_Type', 'ptcarid', 'passenger_avg', '_merge'],
      dtype='str')

In [49]:
punctuality_passengers["Has_Passenger_Count"] = (
    (punctuality_passengers["_merge"] == "both").astype("Int64")
)

In [50]:
(print(punctuality_passengers["Has_Passenger_Count"].sum() == 
       (punctuality_passengers["_merge"] == "both").sum()))

True


In [51]:
punctuality_passengers = (
    punctuality_passengers.drop(columns=["Day_Type", "ptcarid", "_merge"])
)

In [52]:
punctuality_passengers.memory_usage(deep=True).sum() / 1024**3

np.float64(7.9920762134715915)

## 3.5. Adding `SK_Station`

* `Dim_Station` is merged into `punctuality_passengers` to add its surrogate key (`SK_Station`).

* Before the merge, only the relevant `Dim_Station` columns are selected: `SK_Station` and `Ptcar_ID`. 

* The number of `punctuality_passengers` rows before the merge is compared with the number after the merge.  

* Once `SK_Station` is added to `punctuality_passengers`, the redundant columns created by the merge and the `PTCAR_LG_NM_NL` `string` column are dropped. 

* `punctuality_passengers` memory usage is now **7.17 GB**.

In [53]:
dim_station = pd.read_parquet(PATH_PROCESSED / "Dim_Station.parquet")

In [54]:
dim_station = dim_station[["SK_Station", "Ptcar_ID"]]

In [55]:
punctuality_passengers = punctuality_passengers.merge(
    dim_station,
    left_on="PTCAR_NO",
    right_on="Ptcar_ID",
    how="left",
    indicator=True
)

In [56]:
print(len(punctuality_passengers) == punctuality_new_length)

True


In [57]:
punctuality_passengers["_merge"].value_counts()

_merge
both          45541993
left_only            0
right_only           0
Name: count, dtype: int64

In [58]:
punctuality_passengers = (punctuality_passengers.drop(columns=["PTCAR_LG_NM_NL", 
                                                               "PTCAR_NO", 
                                                               "Ptcar_ID", 
                                                               "_merge"])
                                                               )

In [59]:
del dim_station

In [60]:
punctuality_passengers.memory_usage(deep=True).sum() / 1024**3

np.float64(7.213494653813541)

## 3.6. Adding Flag Columns and `Weigthed_Delay`, and Preparing the Load to SQL Server

In [61]:
punctuality_passengers.dtypes

DATDEP                 datetime64[us]
TRAIN_NO                        Int64
LINE_NO_DEP                    string
REAL_TIME_ARR          timedelta64[s]
REAL_TIME_DEP          timedelta64[s]
PLANNED_TIME_ARR       timedelta64[s]
PLANNED_TIME_DEP       timedelta64[s]
DELAY_ARR                     float64
DELAY_DEP                     float64
LINE_NO_ARR                    string
PLANNED_DATE_ARR       datetime64[us]
PLANNED_DATE_DEP       datetime64[us]
REAL_DATE_ARR          datetime64[us]
REAL_DATE_DEP          datetime64[us]
SK_Train_Service                Int64
New_Delay_Arrival      timedelta64[s]
New_Delay_Departure    timedelta64[s]
passenger_avg                   Int64
Has_Passenger_Count             Int64
SK_Station                      Int64
dtype: object

### 3.6.1. Renaming and Converting Delay Columns to `Int64`

* Infrabel's original delay columns are renamed to `Delay_Arr_Infrabel` and `Delay_Dep_Infrabel`, while the new recalculated delay columns are renamed to `Delay_Arr` and `Delay_Dep`.  

* After verification, Infrabel's original delay columns contain only integer numbers, even though they were cast to `float64`. They are converted to `Int64`.  

* The new delay columns are converted from `timedelta64[s]` to `Int64`, with **delays represented as seconds**.  

`DELAY_ARR` and `DELAY_DEP` are converted to `Int64`.

In [62]:
old_delay_cols = ["DELAY_ARR", "DELAY_DEP"]

for col in old_delay_cols:
    print(((punctuality_passengers[col].dropna()) % 1 == 0).all())

for col in old_delay_cols:
    punctuality_passengers[col] = punctuality_passengers[col].astype("Int64")

True
True


`DELAY_ARR` and `DELAY_DEP` are renamed to `Delay_Arr_Infrabel` and `Delay_Dep_Infrabel`.

In [63]:
punctuality_passengers = punctuality_passengers.rename(columns={
    "DELAY_ARR" : "Delay_Arr_Infrabel",
    "DELAY_DEP" : "Delay_Dep_Infrabel"
})

`New_Delay_Arrival` and `New_Delay_Departure` are converted to `Int64` by calculating their total duration in seconds.  

In [64]:
delay_cols = ["New_Delay_Arrival", "New_Delay_Departure"]
for col in delay_cols:
    punctuality_passengers[col] = punctuality_passengers[col].dt.total_seconds().astype("Int64")

`New_Delay_Arrival` and `New_Delay_Departure` are renamed to `Delay_Arr` and `Delay_Dep`.

In [65]:
punctuality_passengers = punctuality_passengers.rename(columns={
    "New_Delay_Arrival" : "Delay_Arr",
    "New_Delay_Departure" : "Delay_Dep"
})

### 3.6.2. Adding Delay Flag Columns

* In this section, four binary delay columns are created:  

    * Two to flag arrival delays using two different thresholds:
        1) `Is_6Min_Late_Arr`, corresponding to Infrabel's original delay threshold for train arrivals at stations.  
        2) `Is_5Min_Late_Arr`, for which a delay is defined as a train arriving more than 5 minutes late at stations.  

    * Two to flag departure delays using two different thresholds:
        1) `Is_6Min_Late_Dep`, corresponding to Infrabel's original delay threshold for train departures from stations.
        2) `Is_5Min_Late_Dep`, for which a delay is defined as a train departing more than 5 minutes late from stations.  

* Both five-minute thresholds (`Is_5Min_Late_Arr` and `Is_5Min_Late_Dep`) correspond to a stricter alternative to Infrabel's original threshold, as explained in the **DESCRIPTION** markdown of this notebook.


In [66]:
punctuality_passengers["Is_6Min_Late_Arr"] = (punctuality_passengers["Delay_Arr"] > 360).astype("Int64")
punctuality_passengers["Is_5Min_Late_Arr"] = (punctuality_passengers["Delay_Arr"] > 300).astype("Int64")
punctuality_passengers["Is_6Min_Late_Dep"] = (punctuality_passengers["Delay_Dep"] > 360).astype("Int64")
punctuality_passengers["Is_5Min_Late_Dep"] = (punctuality_passengers["Delay_Dep"] > 300).astype("Int64")

### 3.6.3. Calculating `Weighted_Delay_Arr` and `Weighted_Delay_Dep`

In this section:

* Two new columns are created: `Weighted_Delay_Arr` and `Weighted_Delay_Dep`.   

* These columns are calculated by **multiplying the arrival and departure delays by the average passenger count** to create two **intermediate delay measures**.  

     - Concretely, arrival and departure delays, expressed in seconds in `Delay_Arr` and `Delay_Dep`, are multiplied by the average passenger count, expressed in passengers in `passenger_avg`, for each row that has a passenger count. The results are expressed in **passenger-seconds** and stored in `Weighted_Delay_Arr` and `Weighted_Delay_Dep`.  

* These intermediate delay measures will be required to calculate the **final passenger-weighted delay metric** of this analysis in Power BI.  

* At the end of this section, the `punctuality_passengers` memory usage is now **9.63 GB**.  

In [67]:
punctuality_passengers["Weighted_Delay_Arr"] = (punctuality_passengers["Delay_Arr"] * punctuality_passengers["passenger_avg"])
punctuality_passengers["Weighted_Delay_Dep"] = (punctuality_passengers["Delay_Dep"] * punctuality_passengers["passenger_avg"])

In [68]:
punctuality_passengers[["Weighted_Delay_Arr", "Weighted_Delay_Dep"]].head()

,Weighted_Delay_Arr,Weighted_Delay_Dep
0,<NA>,271110
1,309145,363700
2,14274,14274
3,35223,35223
4,12597,12597


In [69]:
punctuality_passengers.memory_usage(deep=True).sum() / 1024**3

np.float64(9.673523251898587)

### 3.6.4. Converting `DATDEP` and Time Columns to `Int64`

To prepare them for loading into SQL Server and for use as Foreign Keys in the data warehouse, the service date column (`DATDEP`) and the four time columns are converted to `Int64`.  

The four remaining date columns are kept as `datetime64` and will be loaded as `DATETIME2` columns in SQL Server.  

**⚠️ Warning**: The cell below may take **3 to 4 minutes** to execute.

In [70]:
punctuality_passengers["DATDEP"] = punctuality_passengers["DATDEP"].dt.strftime("%Y%m%d").astype("Int64")
                

In [71]:
columns_to_convert_from_timedelta = ["PLANNED_TIME_ARR", "PLANNED_TIME_DEP", 
                                     "REAL_TIME_ARR", "REAL_TIME_DEP"]

In [72]:
for col_time in columns_to_convert_from_timedelta:
    punctuality_passengers[col_time] = punctuality_passengers[col_time].dt.total_seconds().astype("Int64")

In [73]:
punctuality_passengers.info()

<class 'pandas.DataFrame'>
RangeIndex: 45541993 entries, 0 to 45541992
Data columns (total 26 columns):
 #   Column               Dtype         
---  ------               -----         
 0   DATDEP               Int64         
 1   TRAIN_NO             Int64         
 2   LINE_NO_DEP          string        
 3   REAL_TIME_ARR        Int64         
 4   REAL_TIME_DEP        Int64         
 5   PLANNED_TIME_ARR     Int64         
 6   PLANNED_TIME_DEP     Int64         
 7   Delay_Arr_Infrabel   Int64         
 8   Delay_Dep_Infrabel   Int64         
 9   LINE_NO_ARR          string        
 10  PLANNED_DATE_ARR     datetime64[us]
 11  PLANNED_DATE_DEP     datetime64[us]
 12  REAL_DATE_ARR        datetime64[us]
 13  REAL_DATE_DEP        datetime64[us]
 14  SK_Train_Service     Int64         
 15  Delay_Arr            Int64         
 16  Delay_Dep            Int64         
 17  passenger_avg        Int64         
 18  Has_Passenger_Count  Int64         
 19  SK_Station           Int64    

## 3.7. Refining `Fact_Punctuality`

* The `punctuality_passengers` DataFrame is renamed to `Fact_Punctuality`, then deleted.  

* Its columns are renamed using *Pascal Snake Case* and their order is reorganised to improve clarity.  

In [74]:
Fact_Punctuality = punctuality_passengers

In [75]:
del punctuality_passengers

In [76]:
Fact_Punctuality.columns

Index(['DATDEP', 'TRAIN_NO', 'LINE_NO_DEP', 'REAL_TIME_ARR', 'REAL_TIME_DEP',
       'PLANNED_TIME_ARR', 'PLANNED_TIME_DEP', 'Delay_Arr_Infrabel',
       'Delay_Dep_Infrabel', 'LINE_NO_ARR', 'PLANNED_DATE_ARR',
       'PLANNED_DATE_DEP', 'REAL_DATE_ARR', 'REAL_DATE_DEP',
       'SK_Train_Service', 'Delay_Arr', 'Delay_Dep', 'passenger_avg',
       'Has_Passenger_Count', 'SK_Station', 'Is_6Min_Late_Arr',
       'Is_5Min_Late_Arr', 'Is_6Min_Late_Dep', 'Is_5Min_Late_Dep',
       'Weighted_Delay_Arr', 'Weighted_Delay_Dep'],
      dtype='str')

* Renaming columns

In [77]:
Fact_Punctuality = Fact_Punctuality.rename(columns={
    "DATDEP" : "Date_Service",
    "TRAIN_NO" : "Train_No",
    "LINE_NO_DEP" : "Line_No_Dep",
    "REAL_TIME_ARR" : "Real_Time_Arr",
    "REAL_TIME_DEP" : "Real_Time_Dep",
    "PLANNED_TIME_ARR" : "Planned_Time_Arr",
    "PLANNED_TIME_DEP" : "Planned_Time_Dep",
    "LINE_NO_ARR" : "Line_No_Arr",
    "PLANNED_DATE_ARR" : "Planned_Date_Arr",
    "PLANNED_DATE_DEP" : "Planned_Date_Dep",
    "REAL_DATE_ARR" : "Real_Date_Arr",
    "REAL_DATE_DEP" : "Real_Date_Dep",
    "passenger_avg" : "Passenger_Count_Avg"
})

* Reorganising columns

In [78]:
Fact_Punctuality = (Fact_Punctuality[[
    "Date_Service", "SK_Station", "SK_Train_Service", "Train_No",
    "Planned_Time_Arr", "Real_Time_Arr", "Delay_Arr", "Delay_Arr_Infrabel", "Weighted_Delay_Arr",
    "Planned_Time_Dep", "Real_Time_Dep", "Delay_Dep", "Delay_Dep_Infrabel","Weighted_Delay_Dep",
    "Is_6Min_Late_Arr", "Is_5Min_Late_Arr", "Is_6Min_Late_Dep", "Is_5Min_Late_Dep", 
    "Has_Passenger_Count", "Passenger_Count_Avg", "Line_No_Arr", "Line_No_Dep",
    "Planned_Date_Arr", "Real_Date_Arr", "Planned_Date_Dep", "Real_Date_Dep"
]])

## 3.8. Exporting to Gold Layer

The **fact table** is now ready to be exported to the **processed directory**, as `Fact_Punctuality.parquet`, before being loaded to the SQL data warehouse.

In [79]:
Fact_Punctuality.to_parquet(PATH_PROCESSED / "Fact_Punctuality.parquet")

In [80]:
del Fact_Punctuality

In [81]:
df_to_verify = pd.read_parquet(PATH_PROCESSED / "Fact_Punctuality.parquet")
df_to_verify.head()

,Date_Service,SK_Station,SK_Train_Service,Train_No,Planned_Time_Arr,Real_Time_Arr,Delay_Arr,Delay_Arr_Infrabel,Weighted_Delay_Arr,Planned_Time_Dep,...,Is_6Min_Late_Dep,Is_5Min_Late_Dep,Has_Passenger_Count,Passenger_Count_Avg,Line_No_Arr,Line_No_Dep,Planned_Date_Arr,Real_Date_Arr,Planned_Date_Dep,Real_Date_Dep
0,20240101,190,17,1813,<NA>,<NA>,<NA>,<NA>,<NA>,47340,...,0,0,1,6455,<NA>,50A,NaT,NaT,2024-01-01,2024-01-01
1,20240101,464,17,1813,48120,48137,17,17,309145,48300,...,0,0,1,18185,50A,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
2,20240101,359,17,1813,48480,48558,78,78,14274,48480,...,0,0,1,183,50A,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
3,20240101,597,17,1813,48660,48719,59,59,35223,48660,...,0,0,1,597,50A,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
4,20240101,662,17,1813,48780,48837,57,57,12597,48780,...,0,0,1,221,50A,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01


In [82]:
print(df_to_verify.shape)
df_to_verify.dtypes

(45541993, 26)


Date_Service                    Int64
SK_Station                      Int64
SK_Train_Service                Int64
Train_No                        Int64
Planned_Time_Arr                Int64
Real_Time_Arr                   Int64
Delay_Arr                       Int64
Delay_Arr_Infrabel              Int64
Weighted_Delay_Arr              Int64
Planned_Time_Dep                Int64
Real_Time_Dep                   Int64
Delay_Dep                       Int64
Delay_Dep_Infrabel              Int64
Weighted_Delay_Dep              Int64
Is_6Min_Late_Arr                Int64
Is_5Min_Late_Arr                Int64
Is_6Min_Late_Dep                Int64
Is_5Min_Late_Dep                Int64
Has_Passenger_Count             Int64
Passenger_Count_Avg             Int64
Line_No_Arr                    string
Line_No_Dep                    string
Planned_Date_Arr       datetime64[us]
Real_Date_Arr          datetime64[us]
Planned_Date_Dep       datetime64[us]
Real_Date_Dep          datetime64[us]
dtype: objec